# 09 · Watch a VLM play MiniGrid KeyCorridor

Run a vision-language model from **OpenRouter** on **`MiniGrid-KeyCorridorS3R3-v0`**
through the unified interface, then **watch the rollout animate inline** and **read the
model's reasoning for every move**. No GPU, no weights — just an API key.

KeyCorridor is a genuinely hard embodied task: the agent must explore a corridor of rooms,
**find a hidden key, unlock a door, and pick up a target object**. Actions are *egocentric*
(turn left/right, move forward, pick up, toggle), so the model has to track its own heading
across many steps — much harder than absolute up/down/left/right. The model is driven one
tick at a time via `process(EnvironmentStep)`; we capture each rendered frame and the raw reply.

**Prereq:** an OpenRouter key. Either `export OPENROUTER_API_KEY=...` before launching
Jupyter, or keep it in `~/.config/brainscore/OPENROUTER.API`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))   # unified/ on path
# Load the key: prefer the env var; else read the per-user config file.
if not os.environ.get('OPENROUTER_API_KEY'):
    for p in [os.path.expanduser('~/.config/brainscore/OPENROUTER.API')]:
        if os.path.exists(p):
            os.environ['OPENROUTER_API_KEY'] = open(p).read().strip(); break
assert os.environ.get('OPENROUTER_API_KEY'), 'Set OPENROUTER_API_KEY or ~/.config/brainscore/OPENROUTER.API.'
print('OpenRouter key loaded')

## Build the model

KeyCorridor is **frame-only** (MiniGrid has no text board), so this needs a **vision** model
in `obs_mode='vision'`. `MODEL` is any vision-capable OpenRouter id — swap freely. A few good
picks (cheapest first): `qwen/qwen3-vl-8b-instruct`, `google/gemini-2.5-flash`,
`anthropic/claude-sonnet-4.6`, `google/gemini-3.1-pro-preview`. Frontier models stand the best
chance on this hard task; small models will mostly flail (which is itself the finding).

`history_window=8` gives the model **short-term memory**: each prompt now includes its last 8
actions plus two derived signals — a *"your last action changed nothing, you're blocked"* flag
and a *"you've repeated this action N times"* warning. Without this, a model re-plans from
scratch every tick, executes only the first step of its plan, and loops forever ("move forward,
*then* I'll turn" — but it never turns). The history is what lets it notice it's stuck.

`max_tokens` is set generously so the model can *reason out loud* before its `Action:` line —
that reasoning is what we capture and display below.

In [ ]:
from brainscore_core.model_interface import BrainScoreModel
from brainscore.model_helpers.api_behavioral import build_api_action_fn

MODEL = 'google/gemini-2.5-flash'   # any VISION-capable OpenRouter id

action_fn = build_api_action_fn('openrouter', MODEL, obs_mode='vision',
                                max_tokens=500, history_window=8)
model = BrainScoreModel(identifier=f'openrouter:{MODEL}', model=None,
                        region_layer_map={}, preprocessors={}, action_fn=action_fn)
print('model ready:', model.identifier)

## Play one episode (capturing every frame)

We drive `MiniGrid-KeyCorridorS3R3-v0` through `model.process(EnvironmentStep)`, mirroring
`brainscore.harnesses.gymnasium_harness.play_gym_episode` but keeping each rendered frame so
we can animate it. The observation handed to the model is `{'frame', 'instruction',
'legal_actions'}` — the full god's-eye render, the mission string, and the action menu.

Each tick is one API call, so this is sequential. KeyCorridor is hard; `max_steps` caps the
episode (and the spend). We clear `action_fn.trace` first so it holds exactly this rollout.

In [ ]:
import numpy as np
import gymnasium as gym
import minigrid  # noqa: F401  registers the MiniGrid-* envs
from brainscore_core.model_interface import EnvironmentStep, EnvironmentResponse
from brainscore.harnesses.gymnasium_harness import MINIGRID_ACTIONS

def rollout_minigrid(model, env_id, max_steps=40, seed=0):
    model._action_fn.trace.clear()                       # fresh reasoning log
    env = gym.make(env_id, render_mode='rgb_array')
    try:
        obs, info = env.reset(seed=seed)
        mission = str(obs.get('mission', ''))
        n_actions = int(env.action_space.n)
        frames, actions, labels = [env.render()], [], []
        solved = False
        for t in range(max_steps):
            frame = np.asarray(env.render())
            step = EnvironmentStep(
                observation={'frame': frame, 'instruction': mission,
                             'legal_actions': dict(MINIGRID_ACTIONS)},
                instruction=mission, is_first=(t == 0), step_num=t)
            resp = model.process(step)
            a = int(np.asarray(resp.action).reshape(-1)[0]) % n_actions
            actions.append(a); labels.append(MINIGRID_ACTIONS.get(a, str(a)))
            obs, reward, terminated, truncated, info = env.step(a)
            frames.append(np.asarray(env.render()))
            if terminated or truncated:
                solved = bool(terminated and reward > 0)
                break
        return frames, actions, labels, mission, solved
    finally:
        env.close()

frames, actions, labels, mission, solved = rollout_minigrid(
    model, 'MiniGrid-DoorKey-6x6-v0', max_steps=40, seed=0)
print(f'mission: {mission!r}')
print(f'solved={solved}  steps={len(actions)}  moves={labels}')

## Watch it play

Inline animation — play / pause / scrub the rollout. The agent is the red triangle (its tip
shows its heading); colored squares are doors/keys/the target object.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(3.8, 3.8)); plt.close(fig)
im = ax.imshow(frames[0]); ax.axis('off')
def update(i):
    im.set_data(frames[i])
    if i == 0:
        ax.set_title('start', fontsize=10)
    elif solved and i == len(frames) - 1:
        ax.set_title(f'move {i}: {labels[i-1]}  —  SOLVED', fontsize=10)
    else:
        ax.set_title(f'move {i}: {labels[i-1]}', fontsize=10)
    return [im]
anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=700, blit=False)
HTML(anim.to_jshtml())

## Read the reasoning behind each move

`build_api_action_fn` records the model's **full raw reply** for every tick on
`action_fn.trace` — the reasoning it wrote before the `Action:` line. `fallback=True` flags a
tick where the reply couldn't be parsed and a random legal move was used instead. On a hard
egocentric task this is where you see the model lose track of its heading or the key.

In [ ]:
for rec in model._action_fn.trace:
    label = labels[rec['step']] if rec['step'] < len(labels) else rec['action']
    flag = '  [unparsed -> random]' if rec['fallback'] else ''
    print(f"=== move {rec['step']}: action {rec['action']} ({label}){flag} ===")
    print(rec['response'].strip())
    print()

## Try more

- **Swap the model:** set `MODEL` to any *vision-capable* OpenRouter id and re-run. For the
  best shot at solving, use a frontier VLM (`anthropic/claude-sonnet-4.6`,
  `google/gemini-3.1-pro-preview`, `openai/gpt-5.2`). Catalog: https://openrouter.ai/models
- **Easier / harder MiniGrid envs** (just change the env id in the rollout call):

  | env id | challenge |
  |---|---|
  | `MiniGrid-Empty-8x8-v0` | navigation only |
  | `MiniGrid-DoorKey-6x6-v0` | one key, one door, then goal |
  | `MiniGrid-KeyCorridorS3R3-v0` | hidden key across a corridor of rooms (this notebook) |
  | `MiniGrid-KeyCorridorS6R3-v0` | bigger corridor, longer horizon |
  | `MiniGrid-ObstructedMaze-1Dl-v0` | key hidden in a box behind a blocked door — hardest |

- **Different layout:** change `seed=` in the rollout call for a fresh room arrangement.
- **Give it more room to act:** raise `max_steps` (more API calls — KeyCorridor's optimal is
  a few dozen egocentric moves).
- **Cheaper text-only sandbox:** the absolute-action `GridGameEnv` has an ASCII board, so
  text models (DeepSeek, Llama) can play it in `obs_mode='ascii'`. See the harness
  `brainscore.harnesses.grid_game`.
- **Score it (no animation):** `python ../scripts/vlm_game/play_api_game.py --provider openrouter --model <id> --env MiniGrid-KeyCorridorS3R3-v0 --obs_mode vision --games 10`.